In [64]:
import pickle
from pathlib import Path

MODEL_PATH = Path("../models/som_model_200hz_ver7_full.pkl")

with open(MODEL_PATH, "rb") as f:
    som_model = pickle.load(f)

print("✅ Modell laddad")

print(som_model.get_summary())

✅ Modell laddad
{'x': 30, 'y': 30, 'input_len': 6, 'sigma': 6, 'learning_rate': 0.1, 'decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'topology': 'rectangular', 'activation_distance': 'euclidean', 'sigma_decay_function': 'asymptotic_decay', 'random_seed': 42}


In [65]:
import pandas as pd

df_test = pd.read_csv("test_features_windows/features_49_60_ALL_WINDOWS_features_new_SCR.csv")

print("Rows:", len(df_test))

Rows: 27545


In [66]:
features = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
    "SCR_Rate",
    "EDA_Tonic_log",
    "EDA_Phasic_log"
]

Test_features = df_test[features]
Test_features.describe()

,HR,HRV_RMSSD,HRV_SDNN,SCR_Rate,EDA_Tonic_log,EDA_Phasic_log
count,27545.000000,27545.000000,27545.000000,27545.000000,27545.000000,27545.000000
mean,81.865104,55.086812,60.268448,0.027895,6.135552,-1.836671
std,11.157498,20.761627,27.788024,0.035599,0.343218,1.405089
min,51.683225,9.632677,9.613084,0.000000,6.000000,-4.000000
25%,73.161505,37.668540,41.420053,0.000000,6.000000,-2.838173
50%,81.128334,55.726904,54.856488,0.000000,6.000000,-1.816804
75%,88.354645,71.230112,73.136105,0.064539,6.000000,-0.936796
max,127.562194,121.856302,200.000000,0.200000,8.000000,2.000000


In [67]:
# Ta bort NaN ENDAST i features
df_test_clean = df_test.dropna(subset=features).copy()

print("Före:", len(df_test))
print("Efter:", len(df_test_clean))

Före: 27545
Efter: 27545


In [77]:
X_test = Test_features[features]   # ✔ behåller namn

# Kontroll (ska vara TRUE)
import numpy as np
print("Har NaN:", np.isnan(X_test).any())

# Scaling
X_test = som_model.scaler.transform(X_test)

X_test.shape

Har NaN: HR                False
HRV_RMSSD         False
HRV_SDNN          False
SCR_Rate          False
EDA_Tonic_log     False
EDA_Phasic_log    False
dtype: bool


(27545, 6)

In [78]:
pred_clusters = som_model.predict_cluster(X_test)

df_test_clean["cluster"] = pred_clusters

c:\Users\pat19\OneDrive\Skrivbord\Thesis\CLAS_SOM_project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [80]:
Test_features["cluster"].value_counts(normalize=True)

KeyError: 'cluster'

In [72]:
df_test_clean.groupby("participant")["cluster"].value_counts(normalize=True)

participant  cluster
49           0          0.714783
             2          0.285217
50           0          0.536522
             2          0.463478
51           2          0.908141
             0          0.091859
52           0          0.702609
             2          0.297391
53           0          0.905488
             2          0.094512
54           0          0.633711
             2          0.366289
55           2          0.972783
             0          0.027217
56           2          0.790505
             0          0.209495
57           2          0.586492
             0          0.413508
58           2          0.964286
             0          0.035714
59           0          0.742048
             2          0.257952
60           0          0.833624
             2          0.166376
Name: proportion, dtype: float64

In [73]:
df_test_clean.groupby("cluster").mean()

,HR,HRV_RMSSD,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_diff_std,RR_valid,SCR_Rate,participant,time_sec,EDA_Tonic_log,EDA_Phasic_log
cluster,,,,,,,,,,,,,,,
0,82.155567,57.267074,60.006193,743.166632,59.248178,615.080417,888.929635,40.892405,57.068473,1.0,0.057213,54.379598,1072.304914,6.035847,-1.63676
2,81.588737,53.012358,60.517976,743.964669,59.789872,616.760538,894.330145,40.662274,52.817654,1.0,0.000000,54.609635,1218.489834,6.230418,-2.02688
